# Capstone Project 01 — Configurable Cash-Application Rule Engine

**Acceptance criteria (from the brief):** Rules vary from client to client, so they must be
configurable. The Rule Engine picks up a client's rules, executes them in priority order, and
assigns each record one of six end states.

| Seq | Condition | End State |
|---|---|---|
| 1 | Bank txn uploaded, not matched to Open AR/Remittance | `Open` |
| 2 | Bank Statement & Open AR/Remittance matched **with variance** | `Partial Match` |
| 3 | Bank Statement & Open AR/Remittance **fully matched** | `Closed` |
| 4 | No invoice details in Bank Statement & no Remittance Advice | `UAC` (Un-applied Cash) |
| 5 | No information in Bank Statement at all | `UIC` (Un-Identified Cash) |
| 6 | Ambiguous — needs analyst input | `Query` |

This notebook builds a deterministic rule engine for states 1–5, and for state 6 (`Query`) it
calls a deployed **Azure AI Foundry Agent** (`gpt-5`, with the *Microsoft Learn MCP server* tool
already attached server-side) to draft the clarification message an analyst would send to the
collector/customer — matching Process Flow step "2.1 Reach out to collector/customer".

Credentials are read from `.env` via `foundry_helper.py` (shared with Capstone 02) — nothing is
hardcoded in this notebook.

## 1. Setup

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

from foundry_helper import ask_agent_text, MODEL_DEPLOYMENT_NAME, MCP_SERVER_NAME

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", None)

print(f"Model deployment  : {MODEL_DEPLOYMENT_NAME}")
print(f"MCP tool attached : {MCP_SERVER_NAME}")

Model deployment  : gpt-5
MCP tool attached : Microsoft Learn MCP server


## 2. Sample data — Bank Statement, Open AR, Remittance Advice

Three feeds, per the brief. `remit_df` rows carry a `linked_bank_txn_id` — in a real pipeline that
link comes from matching the *deposit* (date/amount/reference) before line-level invoice matching
starts; we short-circuit that step here since it isn't part of this capstone's scope.

The 10 sample bank transactions are built to exercise every rule (1–6) and every end state,
including one deliberately ambiguous case (`BT-08`: a single PO that legitimately maps to **two**
invoices under a split shipment) to trigger `Query`.

In [3]:
ar_df = pd.DataFrame([
    dict(invoice_number="INV-1001", customer_name="Alpha Traders",  po_number="PO-500", delivery_number="DN-9001", invoice_date="2026-06-01", invoice_amount=10000.00),
    dict(invoice_number="INV-1002", customer_name="Beta Foods",     po_number="PO-501", delivery_number="DN-9002", invoice_date="2026-06-03", invoice_amount=7500.00),
    dict(invoice_number="INV-1003", customer_name="Gamma Retail",   po_number="PO-502", delivery_number="DN-9003", invoice_date="2026-06-05", invoice_amount=4200.00),
    dict(invoice_number="INV-1004", customer_name="Delta Supplies", po_number="PO-503", delivery_number="DN-9004", invoice_date="2026-06-07", invoice_amount=15000.00),
    dict(invoice_number="INV-1005", customer_name="Epsilon Corp",   po_number="PO-504", delivery_number="DN-9005", invoice_date="2026-06-10", invoice_amount=8800.00),
    dict(invoice_number="INV-1006", customer_name="Zeta Inc",       po_number="PO-505", delivery_number="DN-9006", invoice_date="2026-06-11", invoice_amount=3000.00),
    dict(invoice_number="INV-1008", customer_name="Theta Ltd",      po_number="PO-507", delivery_number="DN-9008", invoice_date="2026-06-15", invoice_amount=6000.00),
    dict(invoice_number="INV-1009", customer_name="Theta Ltd",      po_number="PO-507", delivery_number="DN-9009", invoice_date="2026-06-16", invoice_amount=6000.00),
])

bank_df = pd.DataFrame([
    dict(bank_txn_id="BT-01", customer_name="Alpha Traders",  po_number="PO-500", delivery_number=None, invoice_number=None,    invoice_date=None,         amount=10000.00),
    dict(bank_txn_id="BT-02", customer_name="Beta Foods",     po_number=None,     delivery_number="DN-9002", invoice_number=None, invoice_date=None,         amount=7500.00),
    dict(bank_txn_id="BT-03", customer_name="Gamma Retail",   po_number=None,     delivery_number=None,  invoice_number="INV-1003", invoice_date="2026-06-05", amount=4200.00),
    dict(bank_txn_id="BT-04", customer_name="Delta Supplies", po_number=None,     delivery_number=None,  invoice_number="INV-1004", invoice_date=None,        amount=14500.00),
    dict(bank_txn_id="BT-05", customer_name="Epsilon Corp",   po_number=None,     delivery_number=None,  invoice_number=None,    invoice_date=None,         amount=8800.00),
    dict(bank_txn_id="BT-06", customer_name="Zeta Inc",       po_number=None,     delivery_number=None,  invoice_number=None,    invoice_date=None,         amount=3000.00),
    dict(bank_txn_id="BT-07", customer_name="Alpha Traders",  po_number=None,     delivery_number=None,  invoice_number=None,    invoice_date=None,         amount=2500.00),
    dict(bank_txn_id="BT-08", customer_name="Theta Ltd",      po_number="PO-507", delivery_number=None,  invoice_number=None,    invoice_date=None,         amount=6000.00),
    dict(bank_txn_id="BT-09", customer_name=None,             po_number=None,     delivery_number=None,  invoice_number=None,    invoice_date=None,         amount=5000.00),
    dict(bank_txn_id="BT-10", customer_name="Omega LLC",      po_number=None,     delivery_number=None,  invoice_number="INV-9999", invoice_date=None,      amount=1200.00),
])

remit_df = pd.DataFrame([
    dict(remit_id="REM-01", linked_bank_txn_id="BT-05", customer_name="Epsilon Corp", po_number="PO-504", invoice_number="INV-1005", invoice_date="2026-06-10", amount=8800.00),
    dict(remit_id="REM-02", linked_bank_txn_id="BT-06", customer_name="Zeta Inc",     po_number="PO-505", invoice_number="INV-1006", invoice_date=None,        amount=3000.00),
])

display(ar_df)
display(bank_df)
display(remit_df)

,invoice_number,customer_name,po_number,delivery_number,invoice_date,invoice_amount
0,INV-1001,Alpha Traders,PO-500,DN-9001,2026-06-01,10000.0
1,INV-1002,Beta Foods,PO-501,DN-9002,2026-06-03,7500.0
2,INV-1003,Gamma Retail,PO-502,DN-9003,2026-06-05,4200.0
3,INV-1004,Delta Supplies,PO-503,DN-9004,2026-06-07,15000.0
4,INV-1005,Epsilon Corp,PO-504,DN-9005,2026-06-10,8800.0
5,INV-1006,Zeta Inc,PO-505,DN-9006,2026-06-11,3000.0
6,INV-1008,Theta Ltd,PO-507,DN-9008,2026-06-15,6000.0
7,INV-1009,Theta Ltd,PO-507,DN-9009,2026-06-16,6000.0


,bank_txn_id,customer_name,po_number,delivery_number,invoice_number,invoice_date,amount
0,BT-01,Alpha Traders,PO-500,NaN,NaN,NaN,10000.0
1,BT-02,Beta Foods,NaN,DN-9002,NaN,NaN,7500.0
2,BT-03,Gamma Retail,NaN,NaN,INV-1003,2026-06-05,4200.0
3,BT-04,Delta Supplies,NaN,NaN,INV-1004,NaN,14500.0
4,BT-05,Epsilon Corp,NaN,NaN,NaN,NaN,8800.0
5,BT-06,Zeta Inc,NaN,NaN,NaN,NaN,3000.0
6,BT-07,Alpha Traders,NaN,NaN,NaN,NaN,2500.0
7,BT-08,Theta Ltd,PO-507,NaN,NaN,NaN,6000.0
8,BT-09,NaN,NaN,NaN,NaN,NaN,5000.0
9,BT-10,Omega LLC,NaN,NaN,INV-9999,NaN,1200.0


,remit_id,linked_bank_txn_id,customer_name,po_number,invoice_number,invoice_date,amount
0,REM-01,BT-05,Epsilon Corp,PO-504,INV-1005,2026-06-10,8800.0
1,REM-02,BT-06,Zeta Inc,PO-505,INV-1006,NaN,3000.0


## 3. Configurable rules

Each rule is a plain Python `dict` — no code changes needed to add, remove, or reorder a rule for
a given client, only a config edit. `resolve` says how to get from what the bank/remittance line
gives us to a candidate invoice; `bank_required` / `remit_required` say which fields must be
present for the rule to even be attempted (this is what makes the `Open` vs `UAC` distinction
*client-specific*, see §5).

In [4]:
DEFAULT_RULES = [
    dict(priority=1, rule_id="PO_MATCH", source="bank", resolve="via_po",
         bank_required=["po_number", "amount"], compare_date=False,
         description="Customer Name + PO Number + Amount -> resolve invoice via PO, compare to Open AR"),
    dict(priority=2, rule_id="DELIVERY_MATCH", source="bank", resolve="via_delivery",
         bank_required=["delivery_number", "amount"], compare_date=False,
         description="Customer Name + Delivery Number + Amount -> resolve invoice via delivery number"),
    dict(priority=3, rule_id="INVOICE_DATE_MATCH", source="bank", resolve="direct",
         bank_required=["invoice_number", "invoice_date", "amount"], compare_date=True,
         description="Customer Name + Invoice Number + Invoice Date + Amount"),
    dict(priority=4, rule_id="INVOICE_MATCH", source="bank", resolve="direct",
         bank_required=["invoice_number", "amount"], compare_date=False,
         description="Customer Name + Invoice Number + Amount"),
    dict(priority=5, rule_id="REMIT_PO_INVOICE_DATE_MATCH", source="remittance", resolve="via_remittance",
         remit_required=["po_number", "invoice_number", "invoice_date", "amount"], compare_date=True,
         description="3-way: Remittance Customer+PO+Invoice+Date+Amount vs Open AR"),
    dict(priority=6, rule_id="REMIT_PO_INVOICE_MATCH", source="remittance", resolve="via_remittance",
         remit_required=["po_number", "invoice_number", "amount"], compare_date=False,
         description="3-way: Remittance Customer+PO+Invoice+Amount vs Open AR"),
]

# A second client that never sends delivery numbers or partial remittances, and requires exact
# amounts (zero tolerance). Proves the SAME engine behaves differently on the SAME data purely
# from config.
STRICT_CLIENT_RULES = [r for r in DEFAULT_RULES if r["rule_id"] not in ("DELIVERY_MATCH", "REMIT_PO_INVOICE_MATCH")]

CLIENT_RULEBOOK = {
    "DEFAULT": DEFAULT_RULES,
    "CLIENT_STRICT": STRICT_CLIENT_RULES,
}

pd.DataFrame(DEFAULT_RULES).drop(columns=["bank_required", "remit_required"], errors="ignore")

,priority,rule_id,source,resolve,compare_date,description
0,1,PO_MATCH,bank,via_po,False,Customer Name + PO Number + Amount -> resolve ...
1,2,DELIVERY_MATCH,bank,via_delivery,False,Customer Name + Delivery Number + Amount -> re...
2,3,INVOICE_DATE_MATCH,bank,direct,True,Customer Name + Invoice Number + Invoice Date ...
3,4,INVOICE_MATCH,bank,direct,False,Customer Name + Invoice Number + Amount
4,5,REMIT_PO_INVOICE_DATE_MATCH,remittance,via_remittance,True,3-way: Remittance Customer+PO+Invoice+Date+Amo...
5,6,REMIT_PO_INVOICE_MATCH,remittance,via_remittance,False,3-way: Remittance Customer+PO+Invoice+Amount v...


## 4. The rule engine

In [5]:
from dataclasses import dataclass, field
from typing import Optional


def _is_missing(val):
    return val is None or (isinstance(val, float) and pd.isna(val)) or val == ""


@dataclass
class MatchResult:
    end_state: str
    matched_rule_id: Optional[str] = None
    matched_invoice_number: Optional[str] = None
    invoice_amount: Optional[float] = None
    paid_amount: Optional[float] = None
    variance: Optional[float] = None
    candidate_invoices: Optional[list] = field(default_factory=list)
    notes: str = ""


class RuleEngine:
    """Priority-ordered, configurable matcher for Bank Statement vs Open AR (+ Remittance)."""

    def __init__(self, rules, ar_df, remit_df, tolerance=1.00):
        self.rules = sorted(rules, key=lambda r: r["priority"])
        self.ar_df = ar_df
        self.remit_df = remit_df
        self.tolerance = tolerance

    def _lookup_ar(self, **filters):
        mask = pd.Series(True, index=self.ar_df.index)
        for col, val in filters.items():
            mask &= self.ar_df[col] == val
        return self.ar_df[mask]

    def _rule_applicable(self, txn, remit_row, rule):
        if rule["source"] == "bank":
            return not any(_is_missing(txn.get(f)) for f in rule["bank_required"])
        return remit_row is not None and not any(_is_missing(remit_row.get(f)) for f in rule["remit_required"])

    def _candidates_for_rule(self, txn, remit_row, rule):
        if rule["resolve"] == "via_po":
            cands = self._lookup_ar(customer_name=txn["customer_name"], po_number=txn["po_number"])
            return cands, txn["amount"]
        if rule["resolve"] == "via_delivery":
            cands = self._lookup_ar(customer_name=txn["customer_name"], delivery_number=txn["delivery_number"])
            return cands, txn["amount"]
        if rule["resolve"] == "direct":
            cands = self._lookup_ar(customer_name=txn["customer_name"], invoice_number=txn["invoice_number"])
            if rule["compare_date"]:
                cands = cands[cands["invoice_date"] == txn["invoice_date"]]
            return cands, txn["amount"]
        if rule["resolve"] == "via_remittance":
            cands = self._lookup_ar(customer_name=remit_row["customer_name"], po_number=remit_row["po_number"],
                                     invoice_number=remit_row["invoice_number"])
            if rule["compare_date"]:
                cands = cands[cands["invoice_date"] == remit_row["invoice_date"]]
            return cands, remit_row["amount"]
        raise ValueError(f"Unknown resolve strategy: {rule['resolve']}")

    def evaluate_transaction(self, txn: dict) -> MatchResult:
        remit_rows = self.remit_df[self.remit_df["linked_bank_txn_id"] == txn["bank_txn_id"]]
        remit_row = remit_rows.iloc[0].to_dict() if not remit_rows.empty else None

        if _is_missing(txn.get("customer_name")):
            return MatchResult(end_state="UIC", notes="No customer/counterparty information on the bank line.")

        any_rule_applicable = False
        ambiguous = None

        for rule in self.rules:
            if not self._rule_applicable(txn, remit_row, rule):
                continue
            any_rule_applicable = True

            candidates, paid_amount = self._candidates_for_rule(txn, remit_row, rule)
            if len(candidates) == 0:
                continue
            if len(candidates) > 1:
                ambiguous = ambiguous or (rule, candidates, paid_amount)
                continue  # a more specific, lower-priority rule might still resolve this uniquely

            ar_row = candidates.iloc[0]
            variance = round(float(ar_row["invoice_amount"]) - float(paid_amount), 2)
            end_state = "Closed" if abs(variance) <= self.tolerance else "Partial Match"
            note = "Fully matched." if end_state == "Closed" else (
                f"Variance of {variance:+.2f} vs invoice {ar_row['invoice_number']}; "
                "unmatched value should be posted to UAC pending deduction review (see Capstone 02)."
            )
            return MatchResult(
                end_state=end_state, matched_rule_id=rule["rule_id"],
                matched_invoice_number=ar_row["invoice_number"], invoice_amount=float(ar_row["invoice_amount"]),
                paid_amount=float(paid_amount), variance=variance, notes=note,
            )

        if ambiguous is not None:
            rule, candidates, paid_amount = ambiguous
            invs = candidates["invoice_number"].tolist()
            return MatchResult(
                end_state="Query", matched_rule_id=rule["rule_id"], candidate_invoices=invs,
                paid_amount=float(paid_amount),
                notes=f"{len(invs)} invoices matched rule '{rule['rule_id']}' ({', '.join(invs)}); analyst input required.",
            )

        if not any_rule_applicable:
            return MatchResult(end_state="UAC", notes="Customer identified but no PO/Delivery/Invoice reference "
                                                        "usable under this client's active rules, and no remittance advice.")

        return MatchResult(end_state="Open", notes="Bank transaction uploaded; no matching Open AR/Remittance record found yet.")


def run_engine(client_id, bank_df, ar_df, remit_df, tolerance=1.00) -> pd.DataFrame:
    engine = RuleEngine(CLIENT_RULEBOOK[client_id], ar_df, remit_df, tolerance=tolerance)
    rows = []
    for txn in bank_df.to_dict("records"):
        r = engine.evaluate_transaction(txn)
        rows.append(dict(bank_txn_id=txn["bank_txn_id"], customer_name=txn["customer_name"], bank_amount=txn["amount"],
                          end_state=r.end_state, matched_rule=r.matched_rule_id, matched_invoice=r.matched_invoice_number,
                          variance=r.variance, notes=r.notes))
    return pd.DataFrame(rows)


results_df = run_engine("DEFAULT", bank_df, ar_df, remit_df, tolerance=1.00)
results_df

,bank_txn_id,customer_name,bank_amount,end_state,matched_rule,matched_invoice,variance,notes
0,BT-01,Alpha Traders,10000.0,Closed,PO_MATCH,INV-1001,0.0,Fully matched.
1,BT-02,Beta Foods,7500.0,Closed,DELIVERY_MATCH,INV-1002,0.0,Fully matched.
2,BT-03,Gamma Retail,4200.0,Closed,INVOICE_DATE_MATCH,INV-1003,0.0,Fully matched.
3,BT-04,Delta Supplies,14500.0,Partial Match,INVOICE_MATCH,INV-1004,500.0,Variance of +500.00 vs invoice INV-1004; unmat...
4,BT-05,Epsilon Corp,8800.0,Closed,REMIT_PO_INVOICE_DATE_MATCH,INV-1005,0.0,Fully matched.
5,BT-06,Zeta Inc,3000.0,Closed,REMIT_PO_INVOICE_MATCH,INV-1006,0.0,Fully matched.
6,BT-07,Alpha Traders,2500.0,UAC,NaN,NaN,NaN,Customer identified but no PO/Delivery/Invoice...
7,BT-08,Theta Ltd,6000.0,Query,PO_MATCH,NaN,NaN,"2 invoices matched rule 'PO_MATCH' (INV-1008, ..."
8,BT-09,NaN,5000.0,UIC,NaN,NaN,NaN,No customer/counterparty information on the ba...
9,BT-10,Omega LLC,1200.0,Open,NaN,NaN,NaN,Bank transaction uploaded; no matching Open AR...


## 5. Proof the rules are configurable per client

Re-running the *same* engine on the *same* data with `CLIENT_STRICT` (delivery-number and
partial-remittance rules disabled) changes the outcome for `BT-02` and `BT-06` — no code changed,
only the rule config.

In [6]:
strict_results_df = run_engine("CLIENT_STRICT", bank_df, ar_df, remit_df, tolerance=1.00)

comparison = results_df[["bank_txn_id", "end_state"]].merge(
    strict_results_df[["bank_txn_id", "end_state"]], on="bank_txn_id", suffixes=("_default", "_strict")
)
comparison["changed"] = comparison["end_state_default"] != comparison["end_state_strict"]
comparison

,bank_txn_id,end_state_default,end_state_strict,changed
0,BT-01,Closed,Closed,False
1,BT-02,Closed,UAC,True
2,BT-03,Closed,Closed,False
3,BT-04,Partial Match,Partial Match,False
4,BT-05,Closed,Closed,False
5,BT-06,Closed,UAC,True
6,BT-07,UAC,UAC,False
7,BT-08,Query,Query,False
8,BT-09,UIC,UIC,False
9,BT-10,Open,Open,False


## 6. `Query` end state → Azure AI Foundry Agent

For every transaction the engine could not resolve uniquely, ask the deployed `gpt-5` agent
(`FOUNDRY_AGENT_ENDPOINT`) to draft the clarification message an analyst would send — this is
Process Flow step **"2.1 Reach out to collector/customer if not received"** made concrete. The
agent already has the *Microsoft Learn MCP server* tool attached; `ask_agent_text()` (in
`foundry_helper.py`) transparently approves any MCP tool call the model makes along the way.

In [7]:
def draft_query_message(row) -> str:
    prompt = (
        "You are a cash-application analyst assistant. A customer payment could not be "
        "automatically matched to a single invoice. Draft a short, polite email (3-4 sentences) "
        "to the collections contact for the customer below, asking them to confirm which invoice "
        "the payment should be applied against.\n\n"
        f"Customer: {row['customer_name']}\n"
        f"Payment amount: {row['bank_amount']}\n"
        f"Ambiguity details: {row['notes']}\n"
    )
    return ask_agent_text(prompt)


results_df["suggested_query_message"] = None
query_mask = results_df["end_state"] == "Query"
results_df.loc[query_mask, "suggested_query_message"] = results_df[query_mask].apply(draft_query_message, axis=1)

for _, row in results_df[query_mask].iterrows():
    print(f"--- {row['bank_txn_id']} ({row['customer_name']}) ---")
    print(row["suggested_query_message"])
    print()

--- BT-08 (Theta Ltd) ---
Subject: Payment application clarification for Theta Ltd

Hello [Collections Contact Name],
We received a payment of $6,000.00 from Theta Ltd that could not be auto-applied. Our PO_MATCH rule identified two possible invoices—INV-1008 and INV-1009—and analyst review is required. Could you please confirm which invoice this payment should be applied to, or provide remittance details if different? Thank you for your prompt assistance.



## 7. Manager-facing run summary (agent-generated)

A small, low-risk extra: the same agent turns the run's structured results into a plain-English
summary — showing the Foundry agent adding a reporting layer on top of a fully deterministic
engine, rather than being part of the matching logic itself.

In [8]:
summary_prompt = (
    "Summarize this cash-application rule engine run for an AR manager in 3-4 sentences. "
    "Call out the totals per end state and name any transaction that needs urgent attention.\n\n"
    + results_df[["bank_txn_id", "customer_name", "bank_amount", "end_state", "matched_invoice", "variance"]].to_string(index=False)
)
print(ask_agent_text(summary_prompt))

Closed: 33,500; Partial Match: 14,500 (variance 500); UAC: 2,500; Query: 6,000; UIC: 5,000; Open: 1,200. 
Most items are closed (BT-01, BT-02, BT-03, BT-05, BT-06), with one partial match (BT-04) showing a 500 variance against INV-1004. 
Transactions needing urgent attention: BT-09 (UIC, unidentified customer, 5,000), BT-08 (Query, 6,000), BT-04 (Partial Match with 500 variance), and BT-07 (UAC, unapplied cash, 2,500). 
One item remains open: BT-10 (Omega LLC, 1,200).


## 8. Solution summary — requirement → implementation

| Requirement | Implementation | Why |
|---|---|---|
| Rules configurable per client | `CLIENT_RULEBOOK` dict of rule lists (§3); `RuleEngine` accepts any rule list | Onboarding a client is a config edit, not a code change |
| Priority-ordered execution | `RuleEngine.rules` sorted by `priority`; first uniquely-matching rule wins | Mirrors the "Priority" column in the brief |
| 2-way vs 3-way match | `rule["source"]` = `"bank"` (rules 1–4) vs `"remittance"` (rules 5–6) | Encodes whether Remittance Advice participates |
| `Open` / `Partial Match` / `Closed` | Variance vs `tolerance`; 0 vs 1 AR candidates found | Directly encodes end-state table rows 1–3 |
| `UAC` vs `UIC` | Missing customer name → `UIC`; customer known but no rule applicable & no remittance → `UAC` | Encodes rows 4–5; note `UAC` is *client-relative* — see §5 |
| `Query` + analyst hand-off | >1 AR candidate for one rule → `Query`; `draft_query_message()` calls the Foundry agent | Encodes row 6 + Process Flow step "2.1 reach out to collector/customer" |
| Azure AI Foundry Agent usage | `foundry_helper.ask_agent_text()` → Responses API on `FOUNDRY_AGENT_ENDPOINT`, auto-resolves Microsoft Learn MCP approvals | Uses the supplied endpoint/key/model + MCP tool end-to-end |

**Design choices worth calling out:**
- The engine is row-by-row and readable rather than vectorized — appropriate for a rules demo;
  a production version would index `ar_df`/`remit_df` for O(1) lookups at scale.
- The LLM is used only where judgment is genuinely required (drafting the `Query` message, the
  manager summary) — the match/no-match/variance decisions themselves stay 100% deterministic and
  auditable, which is what you want for financial reconciliation.
